## Get the dataset

In [ ]:
from fastai.vision.all import *
import shutil
import os

# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Define your destination in Drive
dest_dir = "/content/drive/MyDrive/CSS/PIDNet/data/VOCdevkit/VOC2012"
if not os.path.exists(dest_dir):
    os.makedirs(dest_dir)

print("Downloading Pascal VOC 2012 from FastAI S3 mirror...")
# This downloads to a temporary spot in Colab (~/.fastai/data)
# URLs.PASCAL_2012 is a stable S3 link
path = untar_data(URLs.PASCAL_2012)

print(f"Data downloaded to: {path}")

# 3. Move/Copy the data to your Drive structure
# The FastAI download extracts to a folder named 'pascal_2012' usually containing 'train', 'val', etc.
# We need to map it to the standard VOC structure: JPEGImages, SegmentationClass, etc.

print("Moving files to your Drive... (this handles the structure automatically)")

# Define source paths from the downloaded content
src_images = path / 'JPEGImages'
src_labels = path / 'SegmentationClass'
src_sets = path / 'ImageSets'

# Define destination paths in your Drive
dest_images = os.path.join(dest_dir, 'JPEGImages')
dest_labels = os.path.join(dest_dir, 'SegmentationClass')
dest_sets = os.path.join(dest_dir, 'ImageSets')

# Function to safely copy folders
def copy_folder(src, dst):
    if os.path.exists(src):
        if os.path.exists(dst):
            print(f"Directory {dst} already exists, merging/skipping...")
        else:
            shutil.copytree(src, dst)
            print(f"Copied {src} -> {dst}")
    else:
        print(f"Warning: Source {src} not found!")

copy_folder(src_images, dest_images)
copy_folder(src_labels, dest_labels)
copy_folder(src_sets, dest_sets)

print("------------------------------------------------")
print("Download and setup complete!")
print(f"Your data is ready at: {dest_dir}")

Mounted at /content/drive


<div><progress max="2618908000" value="2618908672"></progress> 100.00% [2618908672/2618908000 00:42&lt;00:00]</div>

Data downloaded to: /root/.fastai/data/pascal_2012
Moving files to your Drive... (this handles the structure automatically)
------------------------------------------------
Download and setup complete!
Your data is ready at: /content/drive/MyDrive/CSS/PIDNet/data/VOCdevkit/VOC2012


In [ ]:
import os
import shutil

# 1. Define where FastAI saved the data
download_base = "/root/.fastai/data/pascal_2012"
dest_dir = "/content/drive/MyDrive/CSS/PIDNet/data/VOCdevkit/VOC2012"

print(f"Scanning {download_base} for your data...")

# 2. Function to find the real root folder
def find_voc_dirs(start_dir):
    for root, dirs, files in os.walk(start_dir):
        if 'JPEGImages' in dirs and 'SegmentationClass' in dirs:
            return root
    return None

# 3. Find the actual data location
actual_source = find_voc_dirs(download_base)

if actual_source:
    print(f"FOUND data at: {actual_source}")

    # 4. Move the files
    # We move the contents of 'actual_source' to 'dest_dir'

    # Ensure destination exists
    if not os.path.exists(dest_dir):
        os.makedirs(dest_dir)

    for folder_name in ['JPEGImages', 'SegmentationClass', 'ImageSets']:
        src = os.path.join(actual_source, folder_name)
        dst = os.path.join(dest_dir, folder_name)

        if os.path.exists(src):
            if os.path.exists(dst):
                print(f"Folder {dst} already exists. Skipping or merging...")
            else:
                print(f"Moving {folder_name} to Drive...")
                shutil.move(src, dst)
        else:
            print(f"Warning: {folder_name} missing from source.")

    print("\nSUCCESS: Data successfully moved to Google Drive!")
    print(f"Location: {dest_dir}")

else:
    print("ERROR: Could not find 'JPEGImages' inside the download folder.")
    print("Listing contents of download folder to help debug:")
    for root, dirs, files in os.walk(download_base):
        level = root.replace(download_base, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 4 * (level + 1)
        # Limit printed files
        for f in files[:5]:
            print(f"{subindent}{f}")

Scanning /root/.fastai/data/pascal_2012 for your data...
ERROR: Could not find 'JPEGImages' inside the download folder.
Listing contents of download folder to help debug:
pascal_2012/
    valid.json
    train.json
    train.csv
    segmentation/
        2007_008994.png
        2009_000354.png
        2011_000182.png
        2010_003768.png
        2007_005064.png
    train/
        2009_002940.jpg
        2010_005928.jpg
        2008_004462.jpg
        2008_001147.jpg
        2009_004630.jpg
    test/
        2008_000478.jpg
        2008_002391.jpg
        2008_005951.jpg
        2009_002068.jpg
        2008_004958.jpg


In [ ]:
import os
import shutil
from pathlib import Path
from tqdm import tqdm  # specific for progress bars

# 1. Define Paths
source_root = Path("/root/.fastai/data/pascal_2012")
dest_root = Path("/content/drive/MyDrive/CSS/PIDNet/data/VOCdevkit/VOC2012")

# Standard VOC folders we need to create
dest_jpeg = dest_root / "JPEGImages"
dest_seg = dest_root / "SegmentationClass"
dest_sets = dest_root / "ImageSets" / "Segmentation"

print("Creating standard VOC folders...")
for p in [dest_jpeg, dest_seg, dest_sets]:
    p.mkdir(parents=True, exist_ok=True)

# 2. Function to move files with progress bar
def move_files(src_folder, dest_folder, file_ext=None):
    if not src_folder.exists():
        print(f"Skipping {src_folder} (not found)")
        return

    files = [f for f in src_folder.iterdir() if f.is_file()]
    print(f"Moving {len(files)} files from {src_folder.name} to {dest_folder.name}...")

    for f in tqdm(files):
        # Optional: Filter by extension if needed
        if file_ext and f.suffix != file_ext:
            continue

        # Move file (overwrite if exists to ensure we have the latest)
        shutil.copy(str(f), str(dest_folder / f.name))

# 3. Execute the Move
# Move all images (train and test) into the single JPEGImages folder
move_files(source_root / "train", dest_jpeg, ".jpg")
move_files(source_root / "test", dest_jpeg, ".jpg") # Optional, but good to have

# Move all labels into SegmentationClass
move_files(source_root / "segmentation", dest_seg, ".png")

# 4. Generate the List File (Crucial Step!)
# We create train.txt based on the labels we actually have
print("\nGenerating train.txt based on available masks...")
valid_labels = sorted([f.stem for f in dest_seg.iterdir() if f.suffix == '.png'])

list_file_path = dest_sets / "train.txt"
with open(list_file_path, "w") as f:
    for name in valid_labels:
        f.write(f"{name}\n")

print(f"Created train.txt with {len(valid_labels)} entries.")

# 5. Copy the list to your PIDNet list folder as well (for convenience)
pidnet_list_folder = Path("/content/drive/MyDrive/CSS/PIDNet/list/pascal")
pidnet_list_folder.mkdir(parents=True, exist_ok=True)
shutil.copy(str(list_file_path), str(pidnet_list_folder / "train.txt"))

print("\n------------------------------------------------")
print("FIX COMPLETE!")
print(f"Images are in: {dest_jpeg}")
print(f"Labels are in: {dest_seg}")
print(f"List file is in: {pidnet_list_folder / 'train.txt'}")

Creating standard VOC folders...
Moving 11540 files from train to JPEGImages...


100%|██████████| 11540/11540 [01:23<00:00, 137.64it/s]


Moving 10991 files from test to JPEGImages...


100%|██████████| 10991/10991 [01:20<00:00, 136.95it/s]


Moving 2913 files from segmentation to SegmentationClass...


100%|██████████| 2913/2913 [00:18<00:00, 155.69it/s]



Generating train.txt based on available masks...
Created train.txt with 2913 entries.

------------------------------------------------
FIX COMPLETE!
Images are in: /content/drive/MyDrive/CSS/PIDNet/data/VOCdevkit/VOC2012/JPEGImages
Labels are in: /content/drive/MyDrive/CSS/PIDNet/data/VOCdevkit/VOC2012/SegmentationClass
List file is in: /content/drive/MyDrive/CSS/PIDNet/list/pascal/train.txt


In [ ]:
import os
import random
from pathlib import Path

# 1. Define paths
list_folder = Path("/content/drive/MyDrive/CSS/PIDNet/list/pascal")
full_list_path = list_folder / "train.txt" # This is the big list we made earlier
train_list_path = list_folder / "train_split.txt"
val_list_path = list_folder / "val.txt"

# 2. Read the full list of images
with open(full_list_path, "r") as f:
    lines = f.readlines()

# 3. Shuffle and Split (80% Train, 20% Val)
random.seed(42) # Fixed seed so it's the same every time
random.shuffle(lines)

split_idx = int(len(lines) * 0.8)
train_lines = lines[:split_idx]
val_lines = lines[split_idx:]

# 4. Save the new lists
with open(train_list_path, "w") as f:
    f.writelines(train_lines)

with open(val_list_path, "w") as f:
    f.writelines(val_lines)

print(f"Total Images: {len(lines)}")
print(f"Training: {len(train_lines)} (Saved to {train_list_path})")
print(f"Validation: {len(val_lines)} (Saved to {val_list_path})")

Total Images: 2913
Training: 2330 (Saved to /content/drive/MyDrive/CSS/PIDNet/list/pascal/train_split.txt)
Validation: 583 (Saved to /content/drive/MyDrive/CSS/PIDNet/list/pascal/val.txt)


In [ ]:
import os

# Define your paths
data_root = "/content/drive/MyDrive/CSS/PIDNet/data/VOCdevkit/VOC2012"
list_root = "/content/drive/MyDrive/CSS/PIDNet/list/pascal"

img_dir = os.path.join(data_root, "JPEGImages")
mask_dir = os.path.join(data_root, "SegmentationClass")

# The files we need to clean
target_lists = ["train_split.txt", "val.txt"]

def clean_list(filename):
    full_path = os.path.join(list_root, filename)
    if not os.path.exists(full_path):
        print(f"Skipping {filename} (not found)")
        return

    print(f"Checking {filename}...")
    with open(full_path, 'r') as f:
        lines = f.readlines()

    valid_lines = []
    removed_count = 0

    for line in lines:
        name = line.strip()
        if not name: continue

        # Check if BOTH image and label exist
        img_path = os.path.join(img_dir, name + ".jpg")
        mask_path = os.path.join(mask_dir, name + ".png")

        if os.path.exists(img_path) and os.path.exists(mask_path):
            valid_lines.append(line)
        else:
            removed_count += 1
            # Optional: print the first few missing ones to debug
            if removed_count < 5:
                print(f"  Removing missing entry: {name}")

    # Overwrite the file with only valid lines
    with open(full_path, 'w') as f:
        f.writelines(valid_lines)

    print(f"Done! Kept {len(valid_lines)} lines. Removed {removed_count} invalid entries.")

# Run the cleaning
for lst in target_lists:
    clean_list(lst)

Checking train_split.txt...
  Removing missing entry: 2007_006444
  Removing missing entry: 2007_009649
  Removing missing entry: 2007_006409
  Removing missing entry: 2007_009764
Done! Kept 1818 lines. Removed 512 invalid entries.
Checking val.txt...
  Removing missing entry: 2007_005114
  Removing missing entry: 2007_008927
  Removing missing entry: 2007_007902
  Removing missing entry: 2007_006946
Done! Kept 455 lines. Removed 128 invalid entries.


## Train

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install tensorboardX

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.4 MB/s eta 0:00:00


In [ ]:
%cd /content/drive/MyDrive/CSS/PIDNet
!pip install yacs opencv-python-headless easydict pyyaml

/content/drive/MyDrive/CSS/PIDNet


In [ ]:
import os
import tarfile
import zipfile
import shutil
from google.colab import drive

# 1. Mount Drive (To access the Zips)
drive.mount('/content/drive', force_remount=True)

# --- CONFIGURATION ---
# Where the Zips live (Permanent Storage)
ZIP_DIR = "/content/drive/MyDrive/CSS/PIDNet/zips"

# Where we will build the dataset (Temporary but Fast Colab Storage)
# We use /content/data so it is separated from Drive
DATA_ROOT = "/content/data"
VOC_ROOT = os.path.join(DATA_ROOT, "VOCdevkit", "VOC2012")

print(">>> SESSION SETUP: initializing dataset from Zips...")

# 2. Clean/Create Local Directory
if os.path.exists(DATA_ROOT):
    shutil.rmtree(DATA_ROOT)
os.makedirs(VOC_ROOT, exist_ok=True)

# 3. Extract Base Images
voc_tar = os.path.join(ZIP_DIR, "VOCtrainval_11-May-2012.tar")
if not os.path.exists(voc_tar):
    raise FileNotFoundError(f"CRITICAL: Could not find {voc_tar} on Drive!")

print(f"Extracting Images...")
with tarfile.open(voc_tar, 'r') as tar:
    tar.extractall(path=DATA_ROOT)

# 4. Extract Aug Labels (The SBD fix)
aug_zip = os.path.join(ZIP_DIR, "SegmentationClassAug.zip")
if not os.path.exists(aug_zip):
    raise FileNotFoundError(f"CRITICAL: Could not find {aug_zip} on Drive!")

print(f"Extracting Labels...")
with zipfile.ZipFile(aug_zip, 'r') as z:
    z.extractall(path=DATA_ROOT)

# 5. Extract Lists
list_zip = os.path.join(ZIP_DIR, "list.zip")
if not os.path.exists(list_zip):
    raise FileNotFoundError(f"CRITICAL: Could not find {list_zip} on Drive!")

print(f"Extracting Lists...")
with zipfile.ZipFile(list_zip, 'r') as z:
    z.extractall(path=DATA_ROOT)

# 6. Organize (Merge Folders)
print("Organizing...")
# Move Aug Labels to main folder
source_aug = os.path.join(DATA_ROOT, "SegmentationClassAug")
dest_seg = os.path.join(VOC_ROOT, "SegmentationClass")
if os.path.exists(source_aug):
    for f in os.listdir(source_aug):
        shutil.move(os.path.join(source_aug, f), os.path.join(dest_seg, f))

# Move Lists to main folder
source_list = os.path.join(DATA_ROOT, "list")
dest_list = os.path.join(VOC_ROOT, "ImageSets", "Segmentation")
os.makedirs(dest_list, exist_ok=True)

if os.path.exists(source_list):
    shutil.copy(os.path.join(source_list, "train_aug.txt"), os.path.join(dest_list, "train_split.txt"))
    shutil.copy(os.path.join(source_list, "val.txt"), os.path.join(dest_list, "val.txt"))

print("\n✅ DATASET READY at: /content/data/VOCdevkit/VOC2012")

Mounted at /content/drive
>>> SESSION SETUP: initializing dataset from Zips...
Extracting Images...


/tmp/ipython-input-3638272287.py:33: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=DATA_ROOT)


Extracting Labels...
Extracting Lists...
Organizing...

✅ DATASET READY at: /content/data/VOCdevkit/VOC2012


In [ ]:
import os

# --- UPDATED CONFIGURATION (Points to Local Colab Storage) ---
# We are checking /content/data because that is where we extracted the zips
DATA_ROOT = "/content/data/VOCdevkit/VOC2012"

# The lists were also copied to /content/data/VOCdevkit/VOC2012/ImageSets/Segmentation
LIST_ROOT = "/content/data/VOCdevkit/VOC2012/ImageSets/Segmentation"

img_dir = os.path.join(DATA_ROOT, "JPEGImages")
seg_dir = os.path.join(DATA_ROOT, "SegmentationClass")

print(f"--- 1. CHECKING FOLDERS (Local Storage) ---")
# Check JPEGImages
if os.path.exists(img_dir):
    count = len(os.listdir(img_dir))
    print(f" JPEGImages found: {count} files (Expected ~17,125)")
else:
    print(f" JPEGImages MISSING at: {img_dir}")

# Check SegmentationClass
if os.path.exists(seg_dir):
    count = len(os.listdir(seg_dir))
    print(f" SegmentationClass found: {count} files (Expected ~12,031)")
else:
    print(f" SegmentationClass MISSING at: {seg_dir}")


print(f"\n--- 2. CHECKING LISTS ---")
# Note: In the new setup, we renamed train_aug.txt to train_split.txt inside the extraction script
train_list = os.path.join(LIST_ROOT, "train_split.txt")
val_list = os.path.join(LIST_ROOT, "val.txt")

if os.path.exists(train_list):
    num_train = len(open(train_list).readlines())
    print(f" train_split.txt found: {num_train} lines (Expected 10,582)")
else:
    print(f" train_split.txt MISSING at: {train_list}")

if os.path.exists(val_list):
    num_val = len(open(val_list).readlines())
    print(f" val.txt found: {num_val} lines (Expected 1,449)")
else:
    print(f" val.txt MISSING at: {val_list}")


print(f"\n--- 3. ALIGNMENT TEST ---")
if os.path.exists(train_list):
    with open(train_list, 'r') as f:
        sample_id = f.readline().strip()
        sample_id = os.path.splitext(os.path.basename(sample_id))[0]

    print(f"Testing Sample ID: '{sample_id}'")

    img_path = os.path.join(img_dir, sample_id + ".jpg")
    lbl_path = os.path.join(seg_dir, sample_id + ".png")

    if os.path.exists(img_path):
        print(f"  [Image] Found ")
    else:
        print(f"  [Image] MISSING at {img_path} ")

    if os.path.exists(lbl_path):
        print(f"  [Label] Found ")
    else:
        print(f"  [Label] MISSING at {lbl_path} ")
else:
    print("Cannot run Alignment Test because list file is missing.")

--- 1. CHECKING FOLDERS (Local Storage) ---
 JPEGImages found: 17125 files (Expected ~17,125)
 SegmentationClass found: 12031 files (Expected ~12,031)

--- 2. CHECKING LISTS ---
 train_split.txt found: 10582 lines (Expected 10,582)
 val.txt found: 1449 lines (Expected 1,449)

--- 3. ALIGNMENT TEST ---
Testing Sample ID: '2007_000032'
  [Image] Found 
  [Label] Found 


In [ ]:
%cd /content/drive/MyDrive/CSS/PIDNet
!pip install yacs opencv-python-headless easydict pyyaml
!pip install tensorboardX

/content/drive/MyDrive/CSS/PIDNet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.5 MB/s eta 0:00:00


Training s model on VOC

In [ ]:
%cd /content/drive/MyDrive/CSS/PIDNet
!python tools/train.py --cfg configs/pascal/pidnet_s_voc_15.yaml

流式输出内容被截断，只能显示最后 5000 行内容。
710
720
730
740
750
760
770
780
790
800
810
820
830
840
850
860
870
880
890
900
910
920
930
940
950
960
970
980
990
1000
1010
1020
1030
1040
1050
1060
1070
1080
1090
1100
1110
1120
1130
1140
1150
1160
1170
1180
1190
1200
1210
1220
1230
1240
1250
1260
1270
1280
1290
1300
1310
1320
1330
1340
1350
1360
1370
1380
1390
1400
1410
1420
1430
1440
--- Validation Output 0 ---
  Original mIoU: 0.286671
  Metric 1 mIoU: 0.286671 (PixAcc: 0.851008)
  Metric 2 mIoU: 0.286671 (PixAcc: 0.851007)
  Metric 3 mIoU: 0.286671 (PixAcc: 0.851007)
  Original IoU Array: [0.8823 0.4056 0.1968 0.1387 0.1245 0.0713 0.4296 0.3890 0.3843 0.0485
 0.0358 0.0508 0.3119 0.2116 0.3389 0.5673]
---------------------------------
--- Validation Output 1 ---
  Original mIoU: 0.679017
  Metric 1 mIoU: 0.679017 (PixAcc: 0.937073)
  Metric 2 mIoU: 0.679017 (PixAcc: 0.937073)
  Metric 3 mIoU: 0.679017 (PixAcc: 0.937073)
  Original IoU Array: [0.9386 0.7724 0.3814 0.6663 0.5344 0.5779 0.8987 0.7677 0.81

In [ ]:
import yaml
import os

# Path to your config file
config_path = "/content/drive/MyDrive/CSS/PIDNet/configs/pascal/pidnet_s_voc_15.yaml"

print(f"Reading config from: {config_path}")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# --- FORCE THE PATHS TO LOCAL STORAGE ---
print("Updating paths to point to /content/data (Local Colab)...")

config['DATASET']['ROOT'] = "/content/data/VOCdevkit/VOC2012"
config['DATASET']['TRAIN_SET'] = "/content/data/VOCdevkit/VOC2012/ImageSets/Segmentation/train_split.txt"
config['DATASET']['TEST_SET'] = "/content/data/VOCdevkit/VOC2012/ImageSets/Segmentation/val.txt"

# Verify the change
print(f"New ROOT: {config['DATASET']['ROOT']}")

# Save it back
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=None)

print("\n✅ Config updated successfully!")
print("You can now run the training command.")

Reading config from: /content/drive/MyDrive/CSS/PIDNet/configs/pascal/pidnet_s_voc_15.yaml
Updating paths to point to /content/data (Local Colab)...
New ROOT: /content/data/VOCdevkit/VOC2012

✅ Config updated successfully!
You can now run the training command.


Training M model on VOC

In [ ]:
%cd /content/drive/MyDrive/CSS/PIDNet
!python tools/train.py --cfg configs/pascal/pidnet_m_voc_15.yaml

流式输出内容被截断，只能显示最后 5000 行内容。
720
730
740
750
760
770
780
790
800
810
820
830
840
850
860
870
880
890
900
910
920
930
940
950
960
970
980
990
1000
1010
1020
1030
1040
1050
1060
1070
1080
1090
1100
1110
1120
1130
1140
1150
1160
1170
1180
1190
1200
1210
1220
1230
1240
1250
1260
1270
1280
1290
1300
1310
1320
1330
1340
1350
1360
1370
1380
1390
1400
1410
1420
1430
1440
--- Validation Output 0 ---
  Original mIoU: 0.368430
  Metric 1 mIoU: 0.368430 (PixAcc: 0.869159)
  Metric 2 mIoU: 0.368430 (PixAcc: 0.869159)
  Metric 3 mIoU: 0.368430 (PixAcc: 0.869159)
  Original IoU Array: [0.8989 0.5151 0.2351 0.2476 0.2347 0.2019 0.5487 0.4959 0.4638 0.1171
 0.1062 0.1537 0.3557 0.2650 0.4362 0.6191]
---------------------------------
--- Validation Output 1 ---
  Original mIoU: 0.706500
  Metric 1 mIoU: 0.706500 (PixAcc: 0.941999)
  Metric 2 mIoU: 0.706500 (PixAcc: 0.941997)
  Metric 3 mIoU: 0.706500 (PixAcc: 0.941997)
  Original IoU Array: [0.9409 0.8030 0.4062 0.7810 0.5603 0.7003 0.9010 0.7885 0.8404 0

Training L model on VOC

In [ ]:
%cd /content/drive/MyDrive/CSS/PIDNet
!python tools/train.py --cfg configs/pascal/pidnet_l_voc_15.yaml

流式输出内容被截断，只能显示最后 5000 行内容。
710
720
730
740
750
760
770
780
790
800
810
820
830
840
850
860
870
880
890
900
910
920
930
940
950
960
970
980
990
1000
1010
1020
1030
1040
1050
1060
1070
1080
1090
1100
1110
1120
1130
1140
1150
1160
1170
1180
1190
1200
1210
1220
1230
1240
1250
1260
1270
1280
1290
1300
1310
1320
1330
1340
1350
1360
1370
1380
1390
1400
1410
1420
1430
1440
--- Validation Output 0 ---
  Original mIoU: 0.376113
  Metric 1 mIoU: 0.376113 (PixAcc: 0.872575)
  Metric 2 mIoU: 0.376113 (PixAcc: 0.872574)
  Metric 3 mIoU: 0.376113 (PixAcc: 0.872574)
  Original IoU Array: [0.9000 0.5150 0.2334 0.2782 0.2187 0.2478 0.5332 0.4772 0.4569 0.1152
 0.2252 0.1044 0.3638 0.2744 0.4271 0.6472]
---------------------------------
--- Validation Output 1 ---
  Original mIoU: 0.681165
  Metric 1 mIoU: 0.681165 (PixAcc: 0.935756)
  Metric 2 mIoU: 0.681165 (PixAcc: 0.935755)
  Metric 3 mIoU: 0.681165 (PixAcc: 0.935755)
  Original IoU Array: [0.9387 0.8399 0.3894 0.6983 0.6074 0.6505 0.7886 0.6784 0.80